In [9]:
import os
import sys
import logging
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm

sys.path.insert(0, "../src")  # make validation/src importable
from build_amy_epw import AMYEPWBuilder

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

load_dotenv("../../.env")
NREL_API_KEY = os.environ["NREL_API_KEY"]
NREL_EMAIL   = os.environ["NREL_EMAIL"]

YEAR    = 2023
OUT_DIR = Path("../src/data/weather")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Build valid site list from HEMS-RBSA

In [11]:
all_info = pd.read_csv(r"../src/data/HEMS_RBSA/HEMS_RBSA_BASE.csv")
key_columns = [
    "Conditioned_Area", "Qty_Occupants", "Primary_Heating_System_Type",
    "Total_Wall_U-Value", "Window_U-Value", "Window_to_Wall_Ratio",
]
invalid = {"Not Available", "Unknown", "unknown", "not available", "N/A", "n/a", ""}

def is_valid(val):
    if pd.isna(val):
        return False
    return str(val).strip() not in invalid

mask = all_info[key_columns].apply(lambda col: col.map(is_valid)).all(axis=1)
all_info_clean = all_info[mask].copy()
ee_id_rbsa_clean = all_info_clean["ee_site_id"].drop_duplicates().reset_index(drop=True)

print(f"Kept {len(all_info_clean)} / {len(all_info)} rows")
print(f"{len(ee_id_rbsa_clean)} valid sites")

Kept 293 / 377 rows
293 valid sites


## Get unique station_ids for valid sites

In [3]:
sites_meta = pd.read_csv(r"data/metadata/SITES v9.2.csv", dtype={"station_id": str})

valid_station_ids = (
    sites_meta[sites_meta["ee_site_id"].isin(ee_id_rbsa_clean)]
    ["station_id"]
    .dropna()
    .unique()
)

missing = set(ee_id_rbsa_clean) - set(sites_meta["ee_site_id"])
if missing:
    print(f"WARNING: {len(missing)} site(s) have no station mapping: {missing}")

print(f"{len(valid_station_ids)} unique station IDs for {len(ee_id_rbsa_clean)} valid sites")
print(valid_station_ids)

60 unique station IDs for 293 valid sites
<StringArray>
[ '999999-4136', '727856-94176', '727850-24157', '742060-24207',
   '720638-224',   '720202-118', '726813-94195', '725895-94236',
 '726930-24221', '727937-24222', '727870-94119', '727846-24160',
 '727918-94298', '726986-94261',   '720388-469', '727834-24136',
  '720322-4129',   '720923-310', '726881-94273', '726985-24242',
 '727935-24234', '726980-24229', '994350-99999',   '720734-264',
 '726920-24230',  '727945-4205', '727845-24163',  '726836-4201',
 '727810-24243', '726810-24131', '727938-94274', '725970-24225',
 '725866-94178', '720749-24255', '727884-99999', '727924-24223',
 '727730-24153', '726945-24202', '727930-24233', '727934-94248',
 '727855-24114', '727925-94227', '727976-24217', '727857-94129',
 '727790-24146', '742071-24201', '726959-94281',  '999999-4141',
 '726904-24231', '727873-99999', '725785-24145', '726950-24285',
 '727883-24220', '727830-24149',  '726883-4113', '725975-24235',
 '994025-99999', '726940-24232', '

## Build EPW files — one per station_id

In [4]:
errors = {}

for station_id in tqdm(valid_station_ids, desc="Building EPW files"):
    out_path = OUT_DIR / f"{station_id}.epw"
    if out_path.exists():
        print(f"  SKIP  {station_id}  (already exists)")
        continue
    try:
        builder = AMYEPWBuilder(station_id, YEAR, NREL_API_KEY, NREL_EMAIL)
        builder.build(str(out_path))
        print(f"  OK    {station_id} -> {out_path}")
    except Exception as exc:
        errors[station_id] = exc
        print(f"  FAIL  {station_id}: {exc}")

print(f"Done -- {len(valid_station_ids) - len(errors)} succeeded, {len(errors)} failed")
if errors:
    print("Failed station IDs:", list(errors.keys()))

Building EPW files:   0%|          | 0/60 [00:00<?, ?it/s]INFO  === AMYEPWBuilder: station=999999-4136, year=2024 ===
INFO  [1/4] Fetching station metadata
Building EPW files:   2%|▏         | 1/60 [00:00<00:30,  1.92it/s]INFO  === AMYEPWBuilder: station=727856-94176, year=2024 ===
INFO  [1/4] Fetching station metadata


  FAIL  999999-4136: Station 999999-4136 not found in ISD history. Verify the USAF and WBAN portions.


INFO  Station: FELTS FIELD AIRPORT  (47.6860, -117.3210)  elev=595 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 11061 raw observations downloaded
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=47.6860, lon=-117.3210, year=2024
INFO  NSRDB: 8784 rows, TZ offset=-8
INFO  [4/4] Writing EPW → data\weather\727856-94176.epw
INFO  EPW written: data\weather\727856-94176.epw
INFO  Done → data\weather\727856-94176.epw
Building EPW files:   3%|▎         | 2/60 [00:07<03:56,  4.09s/it]INFO  === AMYEPWBuilder: station=727850-24157, year=2024 ===
INFO  [1/4] Fetching station metadata


  OK    727856-94176 -> data\weather\727856-94176.epw


INFO  Station: SPOKANE INTERNATIONAL AIRPORT  (47.6220, -117.5280)  elev=718 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 13954 raw observations downloaded
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=47.6220, lon=-117.5280, year=2024
INFO  NSRDB: 8784 rows, TZ offset=-8
INFO  [4/4] Writing EPW → data\weather\727850-24157.epw
INFO  EPW written: data\weather\727850-24157.epw
INFO  Done → data\weather\727850-24157.epw
Building EPW files:   5%|▌         | 3/60 [00:13<04:41,  4.95s/it]INFO  === AMYEPWBuilder: station=742060-24207, year=2024 ===
INFO  [1/4] Fetching station metadata


...

  OK    726940-24232 -> data\weather\726940-24232.epw


INFO  Station: NORTH BEND MUNICIPAL ARPT  (43.4130, -124.2440)  elev=5 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 13900 raw observations downloaded
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=43.4130, lon=-124.2440, year=2024
INFO  NSRDB: 8784 rows, TZ offset=-8
INFO  [4/4] Writing EPW → data\weather\726917-24284.epw
INFO  EPW written: data\weather\726917-24284.epw
INFO  Done → data\weather\726917-24284.epw
Building EPW files:  98%|█████████▊| 59/60 [04:46<00:05,  5.07s/it]INFO  === AMYEPWBuilder: station=727928-94263, year=2024 ===
INFO  [1/4] Fetching station metadata


  OK    726917-24284 -> data\weather\726917-24284.epw


INFO  Station: BREMERTON NATIONAL AIRPORT  (47.4830, -122.7670)  elev=135 m  UTC-8
INFO  [2/4] Fetching + processing ISD met data
INFO  ISD: 13084 raw observations downloaded
INFO  ISD: resampled to 8760 hourly rows (local time)
INFO  [3/4] Fetching NSRDB solar data
INFO  NSRDB PSM4: lat=47.4830, lon=-122.7670, year=2024
INFO  NSRDB: 8784 rows, TZ offset=-8
INFO  [4/4] Writing EPW → data\weather\727928-94263.epw
INFO  EPW written: data\weather\727928-94263.epw
INFO  Done → data\weather\727928-94263.epw
Building EPW files: 100%|██████████| 60/60 [04:52<00:00,  4.88s/it]

  OK    727928-94263 -> data\weather\727928-94263.epw
Done -- 49 succeeded, 11 failed
Failed station IDs: ['999999-4136', '720638-224', '720202-118', '720388-469', '720322-4129', '720923-310', '720734-264', '727945-4205', '726836-4201', '999999-4141', '726883-4113']
